# RUNX1-FPDMM HSC and RUNX1-mutant AML transcriptomic analyses

This notebook preserves two distinct contrasts against a fixed 817-gene RUNX1/MECOM shared-target set, defined upstream independently of disease differential expression.

| Analysis | Direction and threshold | Statistical unit and background |
| --- | --- | --- |
| FPDMM HSC vs healthy HSC | FPD/healthy; published adjusted P <0.05 and FC <1/1.2 for downregulation | Reuse published cell-level Seurat DE (S9); clean, detected genes also present in the HemeGET model |
| AML D171G/D198G vs RUNX1 truncation | D198G/truncation; raw P <0.05 and log2FC <-1 | Bulk RNA-seq, two patients per group; edgeR expression-filtered genes |

**Nomenclature:** D198G is the source-data/result label used for the manuscript's D171G label (isoform numbering); the notebook retains D198G in source tables. Confirm the transcript/protein accession when reporting an HGVS identifier.

The FPD source cohort comprised 10 patients and 4 healthy donors, but this is not a new 10-vs-4 donor-level pseudobulk analysis. The source DE uses cell-level testing with equal-cell downsampling per donor; those counts do not establish the number of independent HSC DE replicates. `avg_logFC` is natural-log scale, validated here against supplement S2 before conversion to log2.

AML groups are D198G: `04H096`, `06H048`; truncating candidates: `04H138` (Leu98fs), `06H019` (Glu143*). The comparator is truncating AML, not healthy donors or experimentally verified complete-null controls. No gene passed gene-level BH FDR <0.05 in the archived main analysis; nominal enrichment is exploratory. Cytogenetic/patient confounding and gene correlation are not removed by Fisher's test. Neither contrast establishes MECOM dependency or a physical TF interaction.

Archived checkpoints: FPD HSC downregulation has 12,907 background genes, 748 targets and table `[[13,67],[735,12092]]`; AML has 20,459 genes, 749 targets and `[[27,225],[722,19485]]`. Rows here are downregulated / other, columns targets / other genes. The original FPD implementation writes the transpose; the one-sided enrichment result is identical. These are expected historical counts, not newly executed results. Different backgrounds and thresholds preclude comparing their P values as effect sizes.

Dependencies: NumPy, pandas, SciPy (including `scipy.stats.contingency.odds_ratio`), statsmodels, openpyxl, matplotlib. Optional bulk DE additionally needs Rscript, edgeR, limma and statmod. Run with a Python kernel; the R source is embedded and executed via Rscript only when enabled.

## Inputs and execution

FPD requires supplements S2, S9, S10 and S11 from `NIHMS2060404`, the frozen HGNC table, `RUNX1_MECOM_shared817.csv` (`gene`, `RUNX/1`, `EVI1/MECOM`), and the model gene-by-motif matrix with gene symbols as its index. S10/S11 retain the original progenitor/GMP comparisons and the nine-test primary BH correction; do not apply that correction to HSC alone.

AML can start from archived `main_TMM_DE_all.tsv` (`gene`, `logFC`, `PValue`, `FDR`, Boolean `shared817`). To rerun edgeR, also provide reverse-strand raw counts (gene IDs x sample IDs), GENCODE v32 mapping (`gene_id`, `symbol`, `chrom`), `target_sets.tsv` (`gene`, `set`, including `shared_Top3000`), and `sample_inventory.tsv` (`sample_id`, `tier`). The embedded R code retains main TMM plus RLE, added-A134P and expanded-truncation sensitivity analyses, CAMERA tests and descriptive chromosome-7/leave-one-out checks.

Configure all paths. Run the FPD section independently or the AML section independently after configuration. `RUN_BULK_DE=False` uses an existing DE table; set it to True only after configuring the count-level inputs. Each section checks its own inputs. Outputs are separate from inputs; rerunning replaces output files with the same names.

In [ ]:
from pathlib import Path
FPD_SUPPLEMENTS = Path('/path/to/NIHMS2060404')
SHARED_TARGETS = Path('/path/to/RUNX1_MECOM_shared817.csv')
MODEL_MATRIX = Path('/path/to/bulk_hsc_mpp_motif_gene.csv')
HGNC_TABLE = Path('/path/to/hgnc_complete_set.txt')
AML_INPUT = Path('/path/to/aml_inputs')
AML_COUNTS = AML_INPUT / 'candidate_counts_reverse_strand.tsv.gz'
AML_ANNOTATION = AML_INPUT / 'gencode_v32_gene_mapping.tsv'
AML_TARGET_SETS = AML_INPUT / 'target_sets.tsv'
AML_METADATA = AML_INPUT / 'sample_inventory.tsv'
AML_DE_TABLE = AML_INPUT / 'main_TMM_DE_all.tsv'
OUTPUT_ROOT = Path('results/runx1_patient_transcriptomics')
RUN_BULK_DE = False
def require_inputs(paths):
    missing = [str(p) for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError("Configure the input paths first:\n" + "\n".join(missing))

def prepare_output(path, input_roots):
    path = path.resolve()
    for root in input_roots:
        root = root.resolve()
        if path == root or root in path.parents or path in root.parents:
            raise ValueError("Use an output directory separate from all input directories")
    path.mkdir(parents=True, exist_ok=True)

## FPDMM: published HSC differential expression and target enrichment

Primary mode: exact symbols, detected genes and FC 1.2. The retained sensitivity modes harmonize HGNC symbols or include all reported genes. Detection-matched nulls use joint 0.1-width bins for `pct.1` and `pct.2` with 20,000 draws. The primary BH family is three states x three directions; the threshold sensitivity family contains 27 tests per background mode.

The archived HSC downregulation result is enriched against the unrestricted gene background, but not against the detection-matched null: 13 observed shared targets versus 12.84 expected, matched P approximately 0.553 and nine-test BH q approximately 0.818. Report this sensitivity result alongside Fisher enrichment; it limits a target-specific interpretation.


In [ ]:
from pathlib import Path
from collections import defaultdict
import json,hashlib,platform,importlib.metadata,re
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from scipy.stats.contingency import odds_ratio
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

O=OUTPUT_ROOT/'fpd_published_de'
require_inputs([SHARED_TARGETS, MODEL_MATRIX, HGNC_TABLE] +
               [FPD_SUPPLEMENTS/f'NIHMS2060404-supplement-data_file_S{n}.xlsx' for n in [2,9,10,11]])
prepare_output(O, [FPD_SUPPLEMENTS, SHARED_TARGETS.parent, MODEL_MATRIX.parent, HGNC_TABLE.parent])
T=O/'tables';F=O/'figures'
for directory in [T,F,O/'provenance']: directory.mkdir(parents=True,exist_ok=True)
R=SHARED_TARGETS;M=MODEL_MATRIX
ref=pd.read_csv(R);model=pd.read_csv(M,index_col=0).index.astype(str)
assert len(ref)==817 and ref.gene.is_unique and (ref[['RUNX/1','EVI1/MECOM']]>0).all().all()
hgnc=pd.read_csv(HGNC_TABLE,sep='\t',dtype=str).fillna('')
approved=set(hgnc.symbol);prev=defaultdict(set);alias=defaultdict(set)
for r in hgnc.itertuples():
 for col,dct in [('prev_symbol',prev),('alias_symbol',alias)]:
  for old in getattr(r,col).split('|'):
   if old:dct[old].add(r.symbol)
def resolve(g):
 if g in approved:return g,'approved_exact'
 for dictionary,label in [(prev,'previous'),(alias,'alias')]:
  if g in dictionary:
   if len(dictionary[g])==1:return next(iter(dictionary[g])),label
   return g,'ambiguous_not_changed'
 return g,'unmapped_retained'
mapping=[]
raw={};excluded=[];audits=[];input_paths=[R,M,HGNC_TABLE]
for state,n in [('HSC',9),('Progenitor',10),('GMP',11)]:
 f=FPD_SUPPLEMENTS/f'NIHMS2060404-supplement-data_file_S{n}.xlsx';input_paths.append(f)
 d=pd.read_excel(f,header=1);d.columns=d.columns.str.strip();d=d.rename(columns={'Genes':'gene'});d['source_excel_row']=np.arange(len(d))+3
 d['gene']=d.gene.astype(str).str.strip()
 damaged=d.gene.str.contains(r'^\d{1,2}-[A-Za-z]{3}$|^\d{4}-\d{2}-\d{2}',regex=True)
 duplicates=d.gene.duplicated(keep=False)
 bad=damaged|duplicates
 e=d.loc[bad].copy();e['state']=state;e['reason']=np.where(damaged[bad],'date_corrupted_symbol','duplicate_symbol');excluded.append(e)
 clean=d.loc[~bad].copy()
 assert clean.gene.is_unique and np.isfinite(clean[['p_val','avg_logFC','pct.1','pct.2','p_val_adj']]).all().all()
 assert clean.p_val_adj.between(0,1).all() and clean[['pct.1','pct.2']].ge(0).all().all() and clean[['pct.1','pct.2']].le(1).all().all()
 clean['log2FC']=clean.avg_logFC/np.log(2);clean['direction']='FPD_over_Healthy'
 raw[state]=clean
 audits.append(dict(state=state,source_rows=len(d),excluded_rows=int(bad.sum()),clean_rows=len(clean),zero_detected_rows=int((clean[['pct.1','pct.2']].max(axis=1)==0).sum()),reference_exact_available=len(set(ref.gene)&set(clean.gene))))
pd.concat(excluded).to_csv(T/'excluded_corrupt_or_duplicate_rows.csv',index=False)
pd.DataFrame(audits).to_csv(T/'input_audit.csv',index=False)
allnames=set(model)|set(ref.gene)|set.union(*(set(d.gene) for d in raw.values()))
for g in sorted(allnames):
 new,how=resolve(g);mapping.append(dict(original=g,resolved=new,mapping=how))
pd.DataFrame(mapping).to_csv(T/'HGNC_mapping_audit.csv',index=False)
s2=pd.read_excel(FPD_SUPPLEMENTS/'NIHMS2060404-supplement-data_file_S2.xlsx',header=1);s2.columns=s2.columns.str.strip();logqc=[]
for state,d in raw.items():
 j=s2.merge(d,left_on='Genes',right_on='gene');err=float(np.max(np.abs(j[state]-j.log2FC)))
 assert len(j)==40 and err<1e-8
 logqc.append(dict(state=state,n_matched=40,max_error_S2_vs_avg_logFC_div_ln2=err))
pd.DataFrame(logqc).to_csv(T/'log_scale_validation.csv',index=False)
B=20000;seed=20261014;rng=np.random.default_rng(seed)
results=[];coverage=[];strata_audit=[];gene_audits=[];matched_null={}
for mode in ['exact_detected','exact_all_reported','hgnc_detected']:
 harmonize=mode=='hgnc_detected'
 refs={resolve(g)[0] if harmonize else g for g in ref.gene}
 models={resolve(g)[0] if harmonize else g for g in model}
 assert len(refs)==817,'Target mapping collapse must be reviewed'
 for state,original in raw.items():
  d=original.copy();d['original_gene']=d.gene
  if harmonize:
   d.gene=d.gene.map(lambda g:resolve(g)[0]);collision=d.gene.duplicated(keep=False)
   d.loc[collision].to_csv(T/f'{mode}_{state}_mapping_collisions_excluded.csv',index=False);d=d.loc[~collision].copy()
  available=set(d.gene)
  if mode!='exact_all_reported':d=d.loc[d[['pct.1','pct.2']].max(axis=1)>0].copy()
  d=d.loc[d.gene.isin(models)].copy().reset_index(drop=True);d['target817']=d.gene.isin(refs)
  assert d.gene.is_unique
  d.to_csv(T/f'{mode}_{state}_tested_background.csv',index=False)
  for g in ref.gene:
   key=resolve(g)[0] if harmonize else g
   gene_audits.append(dict(mode=mode,state=state,reference_gene=g,matched_gene=key,status='included' if key in set(d.gene) else 'not_detected_or_not_model' if key in available else 'not_in_clean_table'))
  N=len(d);K=int(d.target817.sum())
  coverage.append(dict(mode=mode,state=state,background_N=N,target_K=K,target_not_used=817-K))
  # 10 x 10 fixed-width detection-proportion bins. Hypergeometric draws
  # exactly simulate uniform gene subsets within each bin, without replacement.
  bins1=np.minimum((d['pct.1'].to_numpy()*10).astype(int),9)
  bins2=np.minimum((d['pct.2'].to_numpy()*10).astype(int),9)
  d['bin']=bins1*10+bins2;groups=[]
  for b,s in d.groupby('bin'):
   k=int(s.target817.sum());groups.append((s.index.to_numpy(),k))
   strata_audit.append(dict(mode=mode,state=state,bin=int(b),n_genes=len(s),n_targets=k,pct1_target_mean=s.loc[s.target817,'pct.1'].mean(),pct1_bin_mean=s['pct.1'].mean(),pct2_target_mean=s.loc[s.target817,'pct.2'].mean(),pct2_bin_mean=s['pct.2'].mean()))
  for fold in [1.,1.2,1.5]:
   for direction in ['any','up','down']:
    threshold=np.log(fold)
    effect=d.avg_logFC.abs()>threshold if direction=='any' else d.avg_logFC>threshold if direction=='up' else d.avg_logFC < -threshold
    sig=(d.p_val_adj<0.05)&effect;t=d.target817
    aa=int((sig&t).sum());bb=K-aa;cc=int((sig&~t).sum());dd=N-K-cc
    odds,p=fisher_exact([[aa,bb],[cc,dd]],alternative='greater')
    ci=odds_ratio([[aa,bb],[cc,dd]]).confidence_interval() if K and aa+cc and N>K else None
    null=np.zeros(B,dtype=int);expect=0.
    for idx,k in groups:
     if not k:continue
     n=len(idx);h=int(sig.iloc[idx].sum());expect+=k*h/n
     null+=rng.hypergeometric(h,n-h,k,size=B)
    emp=(1+int((null>=aa).sum()))/(B+1)
    res=dict(mode=mode,state=state,fold_threshold=fold,direction=direction,N=N,K=K,DEG=int(sig.sum()),overlap=aa,target_nonDE=bb,nontarget_DE=cc,nontarget_nonDE=dd,expected=K*int(sig.sum())/N,fold_enrichment=aa/(K*int(sig.sum())/N) if sig.sum() else np.nan,odds_ratio=odds,OR_CI_low=ci.low if ci else np.nan,OR_CI_high=ci.high if ci else np.nan,p_fisher=p,matched_expected=expect,matched_fold=aa/expect if expect else np.nan,p_matched=emp,null_2_5=float(np.quantile(null,.025)),null_97_5=float(np.quantile(null,.975)))
    results.append(res)
    if fold==1.2:
     d.loc[sig&t].to_csv(T/f'{mode}_{state}_{direction}_overlap_FC1.2.csv',index=False)
     matched_null[f'{mode}_{state}_{direction}']=null
res=pd.DataFrame(results)
for mode in res['mode'].unique():
 mask=res['mode'].eq(mode)
 for method in ['fisher','matched']:
  res.loc[mask,f'q_{method}_27']=multipletests(res.loc[mask,f'p_{method}'],method='fdr_bh')[1]
  primary=mask&res.fold_threshold.eq(1.2)
  res.loc[primary,f'q_{method}_primary9']=multipletests(res.loc[primary,f'p_{method}'],method='fdr_bh')[1]
res.to_csv(T/'all_enrichment_tests.csv',index=False)
primary=res.loc[res['mode'].eq('exact_detected')&res.fold_threshold.eq(1.2)].copy()
primary.to_csv(T/'primary_results.csv',index=False)
pd.DataFrame(coverage).to_csv(T/'background_coverage.csv',index=False)
pd.DataFrame(gene_audits).to_csv(T/'reference817_mapping_and_coverage.csv',index=False)
pd.DataFrame(strata_audit).to_csv(T/'detection_matching_strata.csv',index=False)
np.savez_compressed(T/'matched_null_counts_FC1.2.npz',**matched_null)
# White-background manuscript figure: observed versus detection-matched expectation.
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'svg.fonttype':'none','pdf.fonttype':42,'figure.facecolor':'white','axes.facecolor':'white'})
fig,axes=plt.subplots(1,3,figsize=(11.3,3.7),sharey=True)
for ax,state in zip(axes,['HSC','Progenitor','GMP']):
 r=primary.loc[primary.state.eq(state)].set_index('direction').loc[['any','up','down']]
 y=np.arange(3);ax.barh(y-.15,r.overlap,height=.28,color='#2166ac',label='Observed targets')
 ax.barh(y+.15,r.matched_expected,height=.28,color='#bdbdbd',label='Detection-matched expected')
 ax.set_yticks(y,['Any DE','Up in FPD','Down in FPD']);ax.invert_yaxis();ax.set_title(f'{state} (targets = {int(r.K.iloc[0])})');ax.set_xlabel('Number of genes')
 for i,(_,v) in enumerate(r.iterrows()):ax.text(max(v.overlap,v.matched_expected)+1,i,f'q={v.q_matched_primary9:.3g}',va='center',fontsize=9)
 ax.set_xlim(0,max(r.overlap.max(),r.matched_expected.max())*1.6+8)
 ax.spines[['top','right']].set_visible(False)
axes[0].legend(loc='upper center',bbox_to_anchor=(1.6,-.2),ncol=2,frameon=False)
fig.suptitle('RUNX1–MECOM shared targets in published RUNX1-FPD differential expression\nAdjusted P < 0.05; fold change > 1.2; q: detection-matched test, BH across 9 comparisons',fontsize=11)
fig.tight_layout(rect=(0,.08,1,.86))
for ext in ['pdf','svg','png']:fig.savefig(F/f'shared817_published_DE_enrichment.{ext}',dpi=350,bbox_inches='tight')
meta={'status':'completed','input_sha256':{str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in input_paths},'versions':{x:importlib.metadata.version(x) for x in ['numpy','pandas','scipy','statsmodels','openpyxl','matplotlib']},'reference_size':817,'seed':seed,'matched_random_draws':B,'logFC':'natural logarithm; validated against 40 S2 log2FC values per state','primary':'exact_detected; padj<0.05 and |lnFC|>ln(1.2)','background':'clean unique genes reported in DE and HemeGET model; pct.1>0 or pct.2>0','matching':'fixed width 0.1 bins separately on pct.1 and pct.2; sample within joint bins including targets; hypergeometric simulation','multiplicity':'BH across 9 primary state/direction comparisons; separate 27-test correction across thresholds within each background mode','limitations':['published cell-level DE, not donor-level reanalysis','no absolute expression matching: expression means unavailable','no gene-correlation control or patient bootstrap','FPD direction does not establish MECOM or PPI causality'],'HGNC_url':'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt'}
(O/'provenance/analysis_parameters.json').write_text(json.dumps(meta,indent=2))
print(primary[['state','direction','N','K','DEG','overlap','fold_enrichment','p_fisher','q_fisher_primary9','matched_expected','matched_fold','p_matched','q_matched_primary9']].to_string(index=False),flush=True)
print(pd.DataFrame(coverage).to_string(index=False),flush=True)

## AML: optional count-level edgeR analysis

The R program below uses explicitly configured inputs and a separate output root. It is written into that output directory only when `RUN_BULK_DE` is enabled. FDR-based results in this stage remain separate from the nominal-DE Fisher analysis below.

In [ ]:
BULK_R_SOURCE = r'''
args <- commandArgs(trailingOnly=TRUE)
stopifnot(length(args)==5)
root <- normalizePath(args[1])
library(edgeR);library(limma)
set.seed(198)
out <- function(x,name) write.table(x,file.path(root,name),sep='\t',quote=FALSE,row.names=FALSE)
x<-read.delim(gzfile(args[2]),row.names=1,check.names=FALSE)
ann<-read.delim(args[3],check.names=FALSE)
ann<-ann[match(rownames(x),ann$gene_id),];stopifnot(!anyNA(ann$symbol))
# Sum counts across GENCODE gene IDs sharing a symbol, independently of D/L outcomes.
cnts<-rowsum(as.matrix(x),ann$symbol,reorder=FALSE)
map<-aggregate(gene_id~symbol,ann,paste,collapse=';');out(map,'symbol_count_mapping.tsv')
sets<-read.delim(args[4],check.names=FALSE)
setlist<-split(sets$gene,sets$set)
mainD<-c('04H096','06H048');mainL<-c('04H138','06H019')
meta<-read.delim(args[5])
allL<-meta$sample_id[meta$tier=='L_Runt_truncation_candidate']
summaries<-list();cams<-list();fishers<-list()
run <- function(name,D,L,norm='TMM') {
 ids<-c(L,D);group<-factor(c(rep('L',length(L)),rep('D',length(D))),levels=c('L','D'));design<-model.matrix(~group)
 y<-DGEList(cnts[,ids],group=group);keep<-filterByExpr(y,design=design);y<-y[keep,,keep.lib.sizes=FALSE];y<-calcNormFactors(y,method=norm)
 y<-estimateDisp(y,design,robust=TRUE);fit<-glmQLFit(y,design,robust=TRUE);test<-glmQLFTest(fit,coef=2)
 tab<-topTags(test,n=Inf,sort.by='none')$table;tab$gene<-rownames(tab);tab$shared817<-tab$gene%in%setlist$shared_Top3000
 out(tab[order(tab$PValue),],paste0(name,'_DE_all.tsv'));out(tab[tab$shared817,],paste0(name,'_shared817_DE.tsv'))
 out(data.frame(gene=rownames(cnts),tested=keep),paste0(name,'_filter.tsv'))
 out(data.frame(sample=ids,y$samples),paste0(name,'_normalization.tsv'))
 logcpm<-cpm(y,log=TRUE,prior.count=2);out(data.frame(gene=rownames(logcpm),logcpm,check.names=FALSE),paste0(name,'_logCPM.tsv'))
 pdf(file.path(root,paste0(name,'_diagnostics.pdf')),width=7,height=5,bg='white');plotMDS(y,labels=ids,col=ifelse(group=='D','#c73036','#246ba8'));plotBCV(y);plotQLDisp(fit);dev.off()
 v<-voom(y,design,plot=FALSE)
 # Prespecified primary: shared Top3000, fixed inter-gene correlation .01.
 for(rho in c(0.01,0.05,NA)) {
  indices<-lapply(setlist,function(g)which(rownames(y)%in%g));ca<-camera(v,indices,design,contrast=2,inter.gene.cor=rho,sort=FALSE)
  ca$set<-rownames(ca);ca$analysis<-name;ca$rho<-ifelse(is.na(rho),'estimated',as.character(rho));cams[[length(cams)+1]]<<-ca
 }
 for(th in c(0,0.585,1,1.5)) for(dir in c('Down','Up')){
  hit<-tab$FDR<.05 & if(dir=='Down') tab$logFC< -th else tab$logFC>th
  sh<-tab$shared817;a<-sum(hit&sh);b<-sum(hit&!sh);cc<-sum(!hit&sh);d<-sum(!hit&!sh)
  ft<-fisher.test(matrix(c(a,b,cc,d),nrow=2,byrow=TRUE),alternative='greater')
  fishers[[length(fishers)+1]]<<-data.frame(analysis=name,direction=dir,log2FC_threshold=th,target_DE=a,other_DE=b,target_nonDE=cc,other_nonDE=d,enrichment=if(sum(hit)>0) (a/sum(sh))/(sum(hit)/nrow(tab)) else NA,odds_ratio=unname(ft$estimate),PValue=ft$p.value)
 }
 summaries[[name]]<<-data.frame(analysis=name,nD=length(D),nL=length(L),tested=nrow(tab),shared817_tested=sum(tab$shared817),FDR05_down=sum(tab$FDR<.05&tab$logFC<0),FDR05_up=sum(tab$FDR<.05&tab$logFC>0),shared_down_FDR05=sum(tab$shared817&tab$FDR<.05&tab$logFC<0),shared_up_FDR05=sum(tab$shared817&tab$FDR<.05&tab$logFC>0),median_shared_logFC=median(tab$logFC[tab$shared817]),common_dispersion=y$common.dispersion)
 if(name=='main_TMM'){
  saveRDS(list(y=y,fit=fit,test=test,voom=v,design=design),file.path(root,'main_fit.rds'))
  # Nuclear chromosome 7 deletion is a known confound: gene-set sensitivity only.
  chr7<-unique(ann$symbol[ann$chrom=='chr7']);sel<-!rownames(v)%in%chr7
  vv<-v[sel,];idx<-which(rownames(vv)%in%setlist$shared_Top3000)
  ca<-camera(vv,list(shared_Top3000_without_chr7=idx),design,contrast=2,inter.gene.cor=.01);ca$set<-rownames(ca);ca$analysis<-name;ca$rho<-'0.01';cams[[length(cams)+1]]<<-ca
  # Fixed main filter and normalization; descriptive sensitivity, no n=1 inferential p-values.
  d<-logcpm[,'06H048']-rowMeans(logcpm[,mainL,drop=FALSE]);out(data.frame(gene=names(d),delta_logCPM=d,shared817=names(d)%in%setlist$shared_Top3000),'matched_karyotype_1D2L_descriptive.tsv')
  loo<-sapply(ids,function(s){ds<-setdiff(D,s);ls<-setdiff(L,s);rowMeans(logcpm[,ds,drop=FALSE])-rowMeans(logcpm[,ls,drop=FALSE])});out(data.frame(gene=rownames(loo),loo,check.names=FALSE),'leave_one_out_descriptive.tsv')
 }
 cat(name,'done\n')
}
run('main_TMM',mainD,mainL)
run('main_RLE',mainD,mainL,'RLE')
run('D_plus_A134P',c(mainD,'12H151'),mainL)
run('L_including_blood',mainD,allL)
su<-do.call(rbind,summaries);out(su,'analysis_summary.tsv')
ca<-do.call(rbind,lapply(cams,function(z){for(k in c('FDR','Correlation'))if(!k%in%names(z))z[[k]]<-NA_real_;z[,c('NGenes','Direction','PValue','FDR','Correlation','set','analysis','rho')]}));ca$BH_all_sensitivity<-p.adjust(ca$PValue,'BH');out(ca,'camera_all.tsv')
fi<-do.call(rbind,fishers);fi$BH_all_tests<-p.adjust(fi$PValue,'BH');out(fi,'fisher_thresholds.tsv')
capture.output(sessionInfo(),file=file.path(root,'sessionInfo.txt'))
print(su);print(ca[ca$set=='shared_Top3000',])
'''


In [ ]:
import subprocess
if RUN_BULK_DE:
    require_inputs([AML_COUNTS, AML_ANNOTATION, AML_TARGET_SETS, AML_METADATA])
    bulk_output = OUTPUT_ROOT/'aml_bulk_de'
    prepare_output(bulk_output, [AML_INPUT])
    script = bulk_output/'run_bulk_DE.R'
    script.write_text(BULK_R_SOURCE)
    subprocess.run(['Rscript', str(script.resolve()), str(bulk_output.resolve()),
                    str(AML_COUNTS.resolve()), str(AML_ANNOTATION.resolve()),
                    str(AML_TARGET_SETS.resolve()), str(AML_METADATA.resolve())], check=True)
    AML_DE_TABLE = bulk_output/'main_TMM_DE_all.tsv'
else:
    print('Count-level DE skipped; nominal enrichment will use AML_DE_TABLE.')

## AML: nominal downregulation enrichment

One-sided Fisher test, selected versus unselected genes. The reported Fisher P is unadjusted for threshold selection/multiple testing and is distinct from gene-level FDR.

In [ ]:
from pathlib import Path
import pandas as pd
from scipy.stats import fisher_exact
import json
R=OUTPUT_ROOT/'aml_nominal_down_fisher'
require_inputs([AML_DE_TABLE])
prepare_output(R, [AML_DE_TABLE.parent])
x=pd.read_csv(AML_DE_TABLE,sep='\t')
h=(x.PValue<0.05)&(x.logFC < -1);s=x.shared817
assert s.dtype==bool and x.gene.is_unique
A=int((h&s).sum());B=int((h&~s).sum());C=int((~h&s).sum());D=int((~h&~s).sum())
res=fisher_exact([[A,B],[C,D]],alternative='greater')
r={'contrast':'bulk AML D198G vs RUNX1 truncation; 2D vs 2L','threshold':'raw P < 0.05 and log2FC < -1','alternative':'greater','tested_genes':len(x),'shared_tested':int(s.sum()),'downregulated':int(h.sum()),'table':[[A,B],[C,D]],'enrichment_fold':(A/(A+B))/((A+C)/len(x)),'odds_ratio':float(res.statistic),'raw_fisher_P':float(res.pvalue),'expected_overlap':float(h.sum()*s.sum()/len(x)),'interpretation':'Exploratory nominal-DE enrichment; no gene passed gene-level FDR < 0.05. Shared target set is HSC/MPP-derived; RNA contrast is bulk AML.'}
pd.DataFrame([[A,B],[C,D]],index=['Nominal_downregulated','Not_nominal_downregulated'],columns=['Shared_targets','Other_genes']).to_csv(R/'contingency_table.tsv',sep='\t')
x[h].to_csv(R/'nominal_downregulated252.tsv',sep='\t',index=False)
x[h&s].to_csv(R/'shared_downregulated27.tsv',sep='\t',index=False)
(R/'fisher_result.json').write_text(json.dumps(r,indent=2))
print(json.dumps(r,indent=2))

## Source provenance

Adapted from the following analysis scripts. SHA-256 values identify the original local source snapshots, before path portability changes. Source data are not bundled.

| Source script | SHA-256 |
| --- | --- |
| `analyze.py` | `eda1e61477870df6511b35f4344d692a9d4c0944f9d73a9687713d6291ed3f53` |
| `run_bulk_DE.R` | `434b9d90d036238ce3f329644e1bf00f44d22295c2d6f2246c861996390c8b2b` |
| `run_fisher.py` | `eee9b70a06b60b09d10883aecab9d1c79bee69612837b09e4df051a1e68f9ce0` |